# LeetCode 1519 — Number of Nodes in the Sub-Tree With the Same Label bằng Python

Link đề: https://leetcode.com/problems/number-of-nodes-in-the-sub-tree-with-the-same-label/ · Độ khó: **Medium** · Chủ đề: Hash Table, Tree, Depth-First Search, Breadth-First Search, Counting, DP on Trees

Skeleton LeetCode:

```python
class Solution:
    def countSubTrees(self, n: int, edges: list[list[int]], labels: str) -> list[int]:
        ...
```

## 1. Mục tiêu học

1. Biến một **danh sách cạnh vô hướng** thành cây có gốc (gốc = 0): dựng danh sách kề, tìm cha bằng BFS/DFS, **không** giả định `edges[i] = [cha, con]`.
2. Hiểu **DP trên cây**: mỗi nút trả lên cha một mảng đếm 26 chữ cái của cây con của nó (đúng gợi ý của đề).
3. Viết được 4 cách: brute force duyệt từng cây con `O(n²)`, DFS đệ quy trả mảng 26, **duyệt lặp theo thứ tự hậu tố** (tránh `RecursionError` với cây sâu `10⁵`), và mẹo **bộ đếm toàn cục "trước/sau"** chỉ tốn `O(n + 26)` bộ nhớ.
4. Biết vì sao đệ quy sâu trong Python là nguy hiểm và cách thay bằng stack.
5. Viết test có cây ngẫu nhiên, cạnh đảo chiều ngẫu nhiên, và cây "đường thẳng" sâu tối đa.

## 2. Đề bài & ví dụ

Cho một cây (đồ thị vô hướng, liên thông, không chu trình) gồm `n` nút đánh số `0..n−1` và đúng `n − 1` cạnh. **Gốc là nút 0**. Nút `i` có nhãn `labels[i]` (chữ thường). `edges[i] = [aᵢ, bᵢ]` là cạnh nối `aᵢ`–`bᵢ`.

Trả về mảng `ans` độ dài `n`, trong đó `ans[i]` = số nút trong **cây con của nút `i`** có **cùng nhãn** với nút `i` (cây con gồm `i` và mọi hậu duệ — nên `ans[i] ≥ 1`).

**Ví dụ 1:** `n = 7, edges = [[0,1],[0,2],[1,4],[1,5],[2,3],[2,6]], labels = "abaedcd"` → `[2,1,1,1,1,1,1]`.

**Ví dụ 2:** `n = 4, edges = [[0,1],[1,2],[0,3]], labels = "bbbb"` → `[4,2,1,1]`.

**Ví dụ 3:** `n = 5, edges = [[0,1],[0,2],[1,3],[0,4]], labels = "aabab"` → `[3,2,1,1,1]`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `len(edges) = n − 1`, `aᵢ ≠ bᵢ`, `len(labels) = n`, nhãn là chữ thường.

In [1]:
import random
import sys
import time
from collections import deque, Counter
from typing import List

examples = [
    ((7, [[0, 1], [0, 2], [1, 4], [1, 5], [2, 3], [2, 6]], "abaedcd"), [2, 1, 1, 1, 1, 1, 1]),
    ((4, [[0, 1], [1, 2], [0, 3]], "bbbb"), [4, 2, 1, 1]),
    ((5, [[0, 1], [0, 2], [1, 3], [0, 4]], "aabab"), [3, 2, 1, 1, 1]),
    ((4, [[0, 2], [0, 3], [1, 2]], "aeed"), [1, 1, 2, 1]),   # cạnh [1,2]: 1 là CON của 2
]
for args, want in examples:
    print(args, "->", want)

(7, [[0, 1], [0, 2], [1, 4], [1, 5], [2, 3], [2, 6]], 'abaedcd') -> [2, 1, 1, 1, 1, 1, 1]
(4, [[0, 1], [1, 2], [0, 3]], 'bbbb') -> [4, 2, 1, 1]
(5, [[0, 1], [0, 2], [1, 3], [0, 4]], 'aabab') -> [3, 2, 1, 1, 1]
(4, [[0, 2], [0, 3], [1, 2]], 'aeed') -> [1, 1, 2, 1]


Ví dụ 4 tự dựng (kiểm chứng bằng brute force ở mục 5) để nhấn mạnh: cạnh `[1, 2]` **không** có nghĩa 1 là cha của 2 — ở đây 2 gần gốc hơn nên 2 là cha.

## 3. Kiến thức nền

### 3.1 Danh sách kề cho đồ thị vô hướng

In [2]:
def build_adj(n, edges):
    g = [[] for _ in range(n)]
    for a, b in edges:
        g[a].append(b)
        g[b].append(a)
    return g

print(build_adj(4, [[0, 2], [0, 3], [1, 2]]))

[[2, 3], [2], [0, 1], [0]]


### 3.2 Định gốc cây bằng BFS: mảng `parent` và thứ tự duyệt

Từ gốc 0, BFS gán `parent[v] = u` cho mỗi hàng xóm chưa thăm. Thứ tự BFS `order` có tính chất: **cha luôn đứng trước con**. Duyệt `order` **ngược** ⇒ con luôn được xử lý trước cha (thứ tự kiểu hậu tố).

In [3]:
def bfs_order(n, g, root=0):
    parent = [-1] * n
    order = [root]
    seen = [False] * n
    seen[root] = True
    for u in order:                       # list tự lớn dần = hàng đợi
        for v in g[u]:
            if not seen[v]:
                seen[v] = True
                parent[v] = u
                order.append(v)
    return parent, order

p, o = bfs_order(4, build_adj(4, [[0, 2], [0, 3], [1, 2]]))
print("parent =", p, "| order =", o, "| ngược =", o[::-1])

parent = [-1, 2, 0, 0] | order = [0, 2, 3, 1] | ngược = [1, 3, 2, 0]


### 3.3 Chữ cái → chỉ số: `ord(c) - ord('a')`

Mảng đếm 26 phần tử nhanh hơn `dict`/`Counter` khi bảng chữ cái cố định.

In [4]:
for c in "azd":
    print(c, ord(c) - ord("a"))
cnt = [0] * 26
for c in "abaedcd":
    cnt[ord(c) - 97] += 1
print({chr(97 + i): v for i, v in enumerate(cnt) if v})

a 0
z 25
d 3
{'a': 2, 'b': 1, 'c': 1, 'd': 2, 'e': 1}


### 3.4 Giới hạn đệ quy của Python

Mặc định khoảng 1000 tầng. Cây "đường thẳng" `10⁵` nút làm DFS đệ quy vượt giới hạn → `RecursionError`. (Tăng `sys.setrecursionlimit` có thể làm **sập** trình thông dịch do tràn stack C — không an toàn.)

In [5]:
print("recursion limit:", sys.getrecursionlimit())

def depth(k):
    return 0 if k == 0 else 1 + depth(k - 1)

print(depth(500))
try:
    depth(5000)
except RecursionError as e:
    print("RecursionError:", str(e)[:50])

recursion limit: 1000
500
RecursionError: maximum recursion depth exceeded


### 3.5 Cộng hai mảng đếm

Gộp mảng đếm của con vào cha: `for i in range(26): parent_cnt[i] += child_cnt[i]` — `O(26)`.

In [6]:
a = [1, 0, 2] + [0] * 23
b = [0, 3, 1] + [0] * 23
for i in range(26):
    a[i] += b[i]
print(a[:5])

[1, 3, 3, 0, 0]


## 4. Ý tưởng nền tảng / trực giác

Cây Ví dụ 1 (gốc 0):

```
        0(a)
       /    \
    1(b)    2(a)
    /  \    /  \
  4(d) 5(c) 3(e) 6(d)
```

- Cây con của 2 gồm {2, 3, 6} → nhãn `a, e, d` → số nút nhãn `a` = 1.
- Cây con của 0 là cả cây → nhãn `a` xuất hiện ở 0 và 2 → `ans[0] = 2`.

**Nhận xét DP:** mảng đếm nhãn của cây con `u` = (nhãn của `u`) + tổng mảng đếm của các con. Tính **từ lá lên gốc**:

| nút | mảng đếm cây con (chỉ ghi khác 0) | ans |
|---|---|---|
| 4 | d:1 | 1 |
| 5 | c:1 | 1 |
| 1 | b:1, c:1, d:1 | 1 |
| 3 | e:1 | 1 |
| 6 | d:1 | 1 |
| 2 | a:1, d:1, e:1 | 1 |
| 0 | a:2, b:1, c:1, d:2, e:1 | 2 |

In [7]:
args = examples[0][0]
n, edges, labels = args
g = build_adj(n, edges)
parent, order = bfs_order(n, g)
sub = {}
for u in reversed(order):
    c = Counter(labels[u])
    for v in g[u]:
        if v != parent[u]:
            c += sub[v]
    sub[u] = c
    print(f"nút {u}({labels[u]}): {dict(sorted(c.items()))} -> ans={c[labels[u]]}")

nút 6(d): {'d': 1} -> ans=1
nút 3(e): {'e': 1} -> ans=1
nút 5(c): {'c': 1} -> ans=1
nút 4(d): {'d': 1} -> ans=1
nút 2(a): {'a': 1, 'd': 1, 'e': 1} -> ans=1
nút 1(b): {'b': 1, 'c': 1, 'd': 1} -> ans=1
nút 0(a): {'a': 2, 'b': 1, 'c': 1, 'd': 2, 'e': 1} -> ans=2


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** định gốc cây (tìm `parent`), rồi với **mỗi** nút `i`, duyệt toàn bộ cây con của `i` (đi xuống, không đi lên cha) và đếm nút có nhãn `labels[i]`.

In [8]:
def brute(n, edges, labels):
    g = build_adj(n, edges)
    parent, _ = bfs_order(n, g)
    ans = [0] * n
    for i in range(n):
        stack = [i]
        cnt = 0
        while stack:
            u = stack.pop()
            if labels[u] == labels[i]:
                cnt += 1
            for v in g[u]:
                if v != parent[u]:            # chỉ đi xuống con
                    stack.append(v)
        ans[i] = cnt
    return ans

for args, want in examples:
    print(brute(*args), "kỳ vọng", want)

[2, 1, 1, 1, 1, 1, 1] kỳ vọng [2, 1, 1, 1, 1, 1, 1]
[4, 2, 1, 1] kỳ vọng [4, 2, 1, 1]
[3, 2, 1, 1, 1] kỳ vọng [3, 2, 1, 1, 1]
[1, 1, 2, 1] kỳ vọng [1, 1, 2, 1]


**Độ phức tạp:** tổng kích thước mọi cây con = `Σ (độ sâu + 1)` → worst `O(n²)` (cây đường thẳng: `n(n+1)/2 ≈ 5·10⁹` với `n = 10⁵`). Space `O(n)`.

**Nút thắt:** cây con của cha **chứa** cây con của con — ta đếm lại cùng một nút nhiều lần. Hãy để mỗi con "báo cáo" kết quả lên cha (DP).

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (DP trên cây):** `cnt[u] = e(label[u]) + Σ_{v con của u} cnt[v]`, với `e(c)` là vector đơn vị. `ans[u] = cnt[u][label[u]]`. Mỗi nút xử lý `O(26 + số con)` → tổng `O(26n)`.

**Insight 2 (thứ tự xử lý):** chỉ cần con trước cha → đảo ngược thứ tự BFS là đủ, **không cần đệ quy**.

**Insight 3 (bộ đếm toàn cục trước/sau):** duyệt DFS, giữ **một** mảng `cnt[26]` toàn cục đếm các nút đã "vào". Khi **vào** `u`: ghi `before = cnt[label[u]]`, rồi `cnt[label[u]] += 1`. Khi **ra** `u` (đã duyệt xong cây con): `ans[u] = cnt[label[u]] − before`. Vì trong khoảng vào→ra, các nút được thêm vào đúng là các nút của cây con `u`.

**Quy tắc:** *nếu* cây có thể sâu tới `10⁵` *thì* dùng stack/BFS order thay cho đệ quy trong Python.

## 7. Các cách tiếp theo

### Cách 2 — DFS đệ quy trả mảng 26 (minh hoạ, chỉ an toàn với cây nông)

`dfs(u, p)` trả mảng đếm cây con `u`; gọi đệ quy cho mọi hàng xóm `v ≠ p`, cộng dồn.

In [9]:
def dfs_recursive(n, edges, labels):
    g = build_adj(n, edges)
    ans = [0] * n

    def dfs(u, p):
        cnt = [0] * 26
        cnt[ord(labels[u]) - 97] = 1
        for v in g[u]:
            if v != p:
                sub = dfs(v, u)
                for i in range(26):
                    cnt[i] += sub[i]
        ans[u] = cnt[ord(labels[u]) - 97]
        return cnt

    dfs(0, -1)
    return ans

for args, want in examples:
    print(dfs_recursive(*args), "kỳ vọng", want)

[2, 1, 1, 1, 1, 1, 1] kỳ vọng [2, 1, 1, 1, 1, 1, 1]
[4, 2, 1, 1] kỳ vọng [4, 2, 1, 1]
[3, 2, 1, 1, 1] kỳ vọng [3, 2, 1, 1, 1]
[1, 1, 2, 1] kỳ vọng [1, 1, 2, 1]


**Độ phức tạp:** Time `O(26n)`, Space `O(26 · h)` cho các mảng đang sống trên stack + `O(h)` đệ quy (`h` = chiều cao). **Rủi ro:** cây đường thẳng `n > ~1000` → `RecursionError` (xem mục 13).

### Cách 3 — Duyệt lặp theo thứ tự hậu tố (BFS order đảo ngược)

1. BFS từ 0 lấy `parent` và `order`.
2. `cnt[u]` = mảng 26, khởi tạo với nhãn của chính `u`.
3. Duyệt `order` ngược: `ans[u] = cnt[u][label]`; cộng `cnt[u]` vào `cnt[parent[u]]`.

In [10]:
def iterative_postorder(n, edges, labels):
    g = build_adj(n, edges)
    parent, order = bfs_order(n, g)
    lab = [ord(c) - 97 for c in labels]
    cnt = [[0] * 26 for _ in range(n)]
    for u in range(n):
        cnt[u][lab[u]] = 1
    ans = [0] * n
    for u in reversed(order):
        ans[u] = cnt[u][lab[u]]
        p = parent[u]
        if p != -1:
            cp, cu = cnt[p], cnt[u]
            for i in range(26):
                cp[i] += cu[i]
    return ans

for args, want in examples:
    print(iterative_postorder(*args), "kỳ vọng", want)

[2, 1, 1, 1, 1, 1, 1] kỳ vọng [2, 1, 1, 1, 1, 1, 1]
[4, 2, 1, 1] kỳ vọng [4, 2, 1, 1]
[3, 2, 1, 1, 1] kỳ vọng [3, 2, 1, 1, 1]
[1, 1, 2, 1] kỳ vọng [1, 1, 2, 1]


In [11]:
class SolutionPostorder:
    def countSubTrees(self, n: int, edges: list[list[int]], labels: str) -> list[int]:
        g = [[] for _ in range(n)]
        for a, b in edges:
            g[a].append(b)
            g[b].append(a)
        parent = [-1] * n
        seen = [False] * n
        seen[0] = True
        order = [0]
        for u in order:
            for v in g[u]:
                if not seen[v]:
                    seen[v] = True
                    parent[v] = u
                    order.append(v)
        lab = [ord(c) - 97 for c in labels]
        cnt = [[0] * 26 for _ in range(n)]
        ans = [0] * n
        for u in reversed(order):
            cu = cnt[u]
            cu[lab[u]] += 1
            ans[u] = cu[lab[u]]
            p = parent[u]
            if p != -1:
                cp = cnt[p]
                for i in range(26):
                    if cu[i]:
                        cp[i] += cu[i]
        return ans

print([SolutionPostorder().countSubTrees(*a) for a, _ in examples])

[[2, 1, 1, 1, 1, 1, 1], [4, 2, 1, 1], [3, 2, 1, 1, 1], [1, 1, 2, 1]]


**Độ phức tạp:** Time `O(26n)`, Space `O(26n)` (≈ 2.6·10⁶ số nguyên với `n = 10⁵` — chấp nhận được). Không đệ quy → an toàn với mọi độ sâu.

### Cách 4 — Bộ đếm toàn cục "trước/sau" với DFS lặp (vào/ra)

Stack chứa cặp `(u, trạng_thái)`: `0` = vào, `1` = ra. Khi vào: lưu `before[u]`, tăng `cnt`, đẩy `(u, 1)` rồi đẩy các con `(v, 0)`. Khi ra: `ans[u] = cnt[lab[u]] − before[u]`.

In [12]:
def global_counter(n, edges, labels):
    g = build_adj(n, edges)
    lab = [ord(c) - 97 for c in labels]
    cnt = [0] * 26
    before = [0] * n
    ans = [0] * n
    visited = [False] * n
    stack = [(0, 0)]
    while stack:
        u, state = stack.pop()
        if state == 0:
            visited[u] = True
            before[u] = cnt[lab[u]]
            cnt[lab[u]] += 1
            stack.append((u, 1))
            for v in g[u]:
                if not visited[v]:
                    stack.append((v, 0))
        else:
            ans[u] = cnt[lab[u]] - before[u]
    return ans

for args, want in examples:
    print(global_counter(*args), "kỳ vọng", want)

[2, 1, 1, 1, 1, 1, 1] kỳ vọng [2, 1, 1, 1, 1, 1, 1]
[4, 2, 1, 1] kỳ vọng [4, 2, 1, 1]
[3, 2, 1, 1, 1] kỳ vọng [3, 2, 1, 1, 1]
[1, 1, 2, 1] kỳ vọng [1, 1, 2, 1]


Lưu ý: một nút `v` có thể được đẩy `(v, 0)` trước khi được thăm, nhưng trong **cây** mỗi nút chỉ có một đường từ gốc nên chỉ được đẩy một lần (hàng xóm duy nhất đã thăm của nó là cha). Đây là lời giải đẹp: bộ nhớ chỉ `O(n + 26)`.

In [13]:
class SolutionGlobal:
    def countSubTrees(self, n: int, edges: list[list[int]], labels: str) -> list[int]:
        g = [[] for _ in range(n)]
        for a, b in edges:
            g[a].append(b)
            g[b].append(a)
        lab = [ord(c) - 97 for c in labels]
        cnt = [0] * 26
        before = [0] * n
        ans = [0] * n
        visited = [False] * n
        stack = [(0, False)]
        while stack:
            u, leaving = stack.pop()
            if leaving:
                ans[u] = cnt[lab[u]] - before[u]
                continue
            visited[u] = True
            before[u] = cnt[lab[u]]
            cnt[lab[u]] += 1
            stack.append((u, True))
            for v in g[u]:
                if not visited[v]:
                    stack.append((v, False))
        return ans

print([SolutionGlobal().countSubTrees(*a) for a, _ in examples])

[[2, 1, 1, 1, 1, 1, 1], [4, 2, 1, 1], [3, 2, 1, 1, 1], [1, 1, 2, 1]]


**Độ phức tạp:** Time `O(n)` (không có hệ số 26!), Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 4 trên Ví dụ 3 (`edges = [[0,1],[0,2],[1,3],[0,4]]`, `labels = "aabab"`):

| sự kiện | nút(nhãn) | before | cnt sau (a, b) | ans |
|---|---|---|---|---|
| vào | 0(a) | 0 | a1 b0 | |
| vào | 4(b) | 0 | a1 b1 | |
| ra | 4 | | | 1−0 = 1 |
| vào | 2(b) | 1 | a1 b2 | |
| ra | 2 | | | 2−1 = 1 |
| vào | 1(a) | 1 | a2 b2 | |
| vào | 3(a) | 2 | a3 b2 | |
| ra | 3 | | | 3−2 = 1 |
| ra | 1 | | | 3−1 = 2 |
| ra | 0 | | | 3−0 = 3 |

In [14]:
def trace_global(n, edges, labels):
    g = build_adj(n, edges)
    lab = [ord(c) - 97 for c in labels]
    cnt = [0] * 26
    before = [0] * n
    ans = [0] * n
    visited = [False] * n
    stack = [(0, False)]
    while stack:
        u, leaving = stack.pop()
        if leaving:
            ans[u] = cnt[lab[u]] - before[u]
            print(f"ra  {u}({labels[u]}): ans = {cnt[lab[u]]} - {before[u]} = {ans[u]}")
            continue
        visited[u] = True
        before[u] = cnt[lab[u]]
        cnt[lab[u]] += 1
        stack.append((u, True))
        for v in g[u]:
            if not visited[v]:
                stack.append((v, False))
        used = {chr(97 + i): cnt[i] for i in range(26) if cnt[i]}
        print(f"vào {u}({labels[u]}): before={before[u]} cnt={used} stack={[(x, 'ra' if l else 'vào') for x, l in stack]}")
    return ans

print("kết quả:", trace_global(*examples[2][0]))

vào 0(a): before=0 cnt={'a': 1} stack=[(0, 'ra'), (1, 'vào'), (2, 'vào'), (4, 'vào')]
vào 4(b): before=0 cnt={'a': 1, 'b': 1} stack=[(0, 'ra'), (1, 'vào'), (2, 'vào'), (4, 'ra')]
ra  4(b): ans = 1 - 0 = 1
vào 2(b): before=1 cnt={'a': 1, 'b': 2} stack=[(0, 'ra'), (1, 'vào'), (2, 'ra')]
ra  2(b): ans = 2 - 1 = 1
vào 1(a): before=1 cnt={'a': 2, 'b': 2} stack=[(0, 'ra'), (1, 'ra'), (3, 'vào')]
vào 3(a): before=2 cnt={'a': 3, 'b': 2} stack=[(0, 'ra'), (1, 'ra'), (3, 'ra')]
ra  3(a): ans = 3 - 2 = 1
ra  1(a): ans = 3 - 1 = 2
ra  0(a): ans = 3 - 0 = 3
kết quả: [3, 2, 1, 1, 1]


Trace Cách 3 trên Ví dụ 4 (cạnh đảo chiều) — in `parent`, `order` và mảng đếm khi gộp:

In [15]:
def trace_postorder(n, edges, labels):
    g = build_adj(n, edges)
    parent, order = bfs_order(n, g)
    print("parent =", parent, "| order =", order)
    lab = [ord(c) - 97 for c in labels]
    cnt = [[0] * 26 for _ in range(n)]
    ans = [0] * n
    for u in reversed(order):
        cnt[u][lab[u]] += 1
        ans[u] = cnt[u][lab[u]]
        nz = {chr(97 + i): cnt[u][i] for i in range(26) if cnt[u][i]}
        print(f"xử lý {u}({labels[u]}): cây con={nz} ans={ans[u]} -> gộp vào cha {parent[u]}")
        if parent[u] != -1:
            for i in range(26):
                cnt[parent[u]][i] += cnt[u][i]
    return ans

print("kết quả:", trace_postorder(*examples[3][0]))

parent = [-1, 2, 0, 0] | order = [0, 2, 3, 1]
xử lý 1(e): cây con={'e': 1} ans=1 -> gộp vào cha 2
xử lý 3(d): cây con={'d': 1} ans=1 -> gộp vào cha 0
xử lý 2(e): cây con={'e': 2} ans=2 -> gộp vào cha 0
xử lý 0(a): cây con={'a': 1, 'd': 1, 'e': 2} ans=1 -> gộp vào cha -1
kết quả: [1, 1, 2, 1]


## 9. Code nộp LeetCode

Bản chính: bộ đếm toàn cục với DFS lặp (Cách 4) — `O(n)`, không đệ quy.

In [16]:
class Solution:
    def countSubTrees(self, n: int, edges: list[list[int]], labels: str) -> list[int]:
        g = [[] for _ in range(n)]
        for a, b in edges:
            g[a].append(b)
            g[b].append(a)
        lab = [ord(c) - 97 for c in labels]
        cnt = [0] * 26
        before = [0] * n
        ans = [0] * n
        visited = [False] * n
        stack = [(0, False)]
        while stack:
            u, leaving = stack.pop()
            if leaving:
                ans[u] = cnt[lab[u]] - before[u]
                continue
            visited[u] = True
            before[u] = cnt[lab[u]]
            cnt[lab[u]] += 1
            stack.append((u, True))
            for v in g[u]:
                if not visited[v]:
                    stack.append((v, False))
        return ans

for args, want in examples:
    assert Solution().countSubTrees(*args) == want
print("OK ví dụ đề")

OK ví dụ đề


## 10. Vì sao thuật toán đúng

**Cách 3 (DP):** quy nạp theo thứ tự xử lý (ngược BFS). Khi xử lý `u`, mọi con `v` của `u` đứng sau `u` trong BFS nên đã được xử lý và đã cộng `cnt[v]` (= mảng đếm đúng của cây con `v`) vào `cnt[u]`. Cộng thêm nhãn của `u` ⇒ `cnt[u]` là mảng đếm đúng của cây con `u`, do cây con `u` = `{u}` ∪ (hợp rời các cây con của con).

**Cách 4 (trước/sau):** trong DFS, sự kiện "vào" của các nút thuộc cây con `u` xảy ra **đúng** trong khoảng giữa "vào `u`" và "ra `u`", và không nút nào ngoài cây con `u` được vào trong khoảng đó (stack LIFO: `(u, ra)` chỉ được pop khi mọi thứ đẩy sau nó — chính là cây con — đã xử lý xong). Vậy `cnt[label[u]]` tại lúc ra trừ giá trị ngay trước khi vào `u` = số nút nhãn `label[u]` trong cây con `u` (tính cả `u`). ∎

**Định gốc đúng:** vì đồ thị là cây, BFS từ 0 cho mỗi nút đúng một cha — không phụ thuộc chiều ghi của cạnh.

## 11. Phân tích Time / Space

`n ≤ 10⁵`, `Σ bậc = 2(n − 1)`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(Σ kích thước cây con)` = `O(n·h)`, worst `O(n²)` | `O(n)` |
| 2. DFS đệ quy | `O(26n)` | `O(26h + n)`; đệ quy sâu `h` |
| 3. Hậu tố lặp | `O(26n)` | `O(26n)` |
| 4. Bộ đếm trước/sau | `O(n)` | `O(n)` |

`h` = chiều cao cây: cây ngẫu nhiên `h ≈ O(log n)`–`O(√n)`, cây đường thẳng `h = n`.

## 12. Bộ test

Ví dụ đề + ví dụ tự dựng, edge case (`n = 1`, đường thẳng, hình sao, mọi nhãn giống nhau), random (cạnh đảo chiều và xáo trộn, seed cố định) so với brute force, và test lớn `n = 10⁵` (cây ngẫu nhiên + cây đường thẳng sâu `10⁵`).

In [17]:
METHODS = {
    "recursive": lambda a: dfs_recursive(*a),
    "postorder": lambda a: SolutionPostorder().countSubTrees(*a),
    "global": lambda a: Solution().countSubTrees(*a),
}
edge = [
    ((1, [], "z"), [1]),
    ((3, [[2, 1], [1, 0]], "aaa"), [3, 2, 1]),
    ((5, [[0, 1], [0, 2], [0, 3], [0, 4]], "abcda"), [2, 1, 1, 1, 1]),
    ((6, [[0, 1], [1, 2], [2, 3], [3, 4], [4, 5]], "ababab"), [3, 3, 2, 2, 1, 1]),
]
for args, want in examples + edge:
    assert brute(*args) == want, args
    for name, f in METHODS.items():
        assert f(args) == want, (name, args)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [18]:
def random_tree(rng, n, alphabet):
    perm = list(range(1, n))
    rng.shuffle(perm)
    nodes = [0] + perm                    # thứ tự gắn vào cây, gốc 0 trước
    edges = []
    for i in range(1, n):
        u, p = nodes[i], nodes[rng.randrange(i)]
        edges.append([u, p] if rng.random() < 0.5 else [p, u])
    rng.shuffle(edges)
    labels = "".join(rng.choice(alphabet) for _ in range(n))
    return n, edges, labels

rng = random.Random(1519)
for _ in range(400):
    args = random_tree(rng, rng.randint(1, 30), rng.choice(["ab", "abc", "abcdefghijklmnopqrstuvwxyz"]))
    ref = brute(*args)
    for name, f in METHODS.items():
        assert f(args) == ref, (name, args)
print("400 cây ngẫu nhiên: mọi cách khớp brute")

400 cây ngẫu nhiên: mọi cách khớp brute


In [19]:
big_rand = random_tree(random.Random(7), 10**5, "abcde")
n = 10**5
chain = (n, [[i, i + 1] if i % 2 else [i + 1, i] for i in range(n - 1)], "ab" * (n // 2))
for name in ["postorder", "global"]:
    t = time.perf_counter()
    r1 = METHODS[name](big_rand)
    r2 = METHODS[name](chain)
    print(f"{name:9s}: {time.perf_counter() - t:.2f}s | chain ans[:4]={r2[:4]} ans[-2:]={r2[-2:]}")
assert METHODS["postorder"](big_rand) == METHODS["global"](big_rand)
assert METHODS["global"](chain)[0] == n // 2 and METHODS["global"](chain)[-1] == 1

postorder: 0.46s | chain ans[:4]=[50000, 50000, 49999, 49999] ans[-2:]=[1, 1]


global   : 0.22s | chain ans[:4]=[50000, 50000, 49999, 49999] ans[-2:]=[1, 1]


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi `edges[i] = [cha, con]`

Đề không hứa chiều nào; Ví dụ 4 có cạnh `[1, 2]` trong đó 2 mới là cha.

In [20]:
def wrong_directed(n, edges, labels):
    children = [[] for _ in range(n)]
    for a, b in edges:
        children[a].append(b)                      # SAI: giả định a là cha
    ans = [0] * n
    def count(u, c):
        return (labels[u] == c) + sum(count(v, c) for v in children[u])
    for i in range(n):
        ans[i] = count(i, labels[i])
    return ans

args, want = examples[3]
print("sai:", wrong_directed(*args), "| đúng:", want)

sai: [1, 2, 1, 1] | đúng: [1, 1, 2, 1]


### Lỗi 2 — DFS đệ quy trên cây sâu → `RecursionError`

Đúng trên ví dụ nhỏ, nhưng cây đường thẳng 5000 nút đã vượt giới hạn (ta bắt exception để in).

In [21]:
n = 5000
deep = (n, [[i, i + 1] for i in range(n - 1)], "a" * n)
try:
    dfs_recursive(*deep)
    print("không lỗi?")
except RecursionError as e:
    print("sai: RecursionError ->", str(e)[:45])
print("đúng (lặp):", Solution().countSubTrees(*deep)[:3], "...")

sai: RecursionError -> maximum recursion depth exceeded
đúng (lặp): [5000, 4999, 4998] ...


### Lỗi 3 — Quên bỏ qua cha khi duyệt đồ thị vô hướng

Không có `v != p` / `visited` → đi ngược lên cha → vòng lặp vô hạn (ở đây đệ quy vô hạn, bắt `RecursionError`); với phiên bản giới hạn bước, kết quả bị đếm trùng.

In [22]:
def wrong_no_parent_check(n, edges, labels, limit=50):
    g = build_adj(n, edges)
    ans = [0] * n
    steps = 0
    def dfs(u):
        nonlocal steps
        steps += 1
        if steps > limit:                           # chặn để không chạy mãi
            raise RecursionError("đi vòng qua lại cha-con")
        cnt = Counter(labels[u])
        for v in g[u]:                              # SAI: không loại cha
            cnt += dfs(v)
        ans[u] = cnt[labels[u]]
        return cnt
    dfs(0)
    return ans

args, want = examples[1]
try:
    print("sai:", wrong_no_parent_check(*args))
except RecursionError as e:
    print("sai: RecursionError ->", e, "| đúng:", want)

sai: RecursionError -> đi vòng qua lại cha-con | đúng: [4, 2, 1, 1]


### Lỗi 4 — Tạo mảng đếm bằng `[[0] * 26] * n`

Mọi nút dùng chung **một** mảng → kết quả loạn.

In [23]:
def wrong_alias(n, edges, labels):
    g = build_adj(n, edges)
    parent, order = bfs_order(n, g)
    lab = [ord(c) - 97 for c in labels]
    cnt = [[0] * 26] * n                           # SAI: n tham chiếu tới cùng 1 list
    ans = [0] * n
    for u in reversed(order):
        cnt[u][lab[u]] += 1
        ans[u] = cnt[u][lab[u]]
        if parent[u] != -1:
            for i in range(26):
                cnt[parent[u]][i] += cnt[u][i]
    return ans

for args, want in examples[:2]:
    print("sai:", wrong_alias(*args), "| đúng:", want)

sai: [5, 1, 1, 1, 9, 1, 1] | đúng: [2, 1, 1, 1, 1, 1, 1]
sai: [15, 7, 1, 3] | đúng: [4, 2, 1, 1]


### Lỗi 5 — Quên tính chính nút `i`

Đề nói "cây con gồm chính nút đó" → `ans[i] ≥ 1`. Đếm chỉ các hậu duệ là lệch 1.

In [24]:
def wrong_exclude_self(n, edges, labels):
    g = build_adj(n, edges)
    parent, order = bfs_order(n, g)
    lab = [ord(c) - 97 for c in labels]
    cnt = [[0] * 26 for _ in range(n)]
    ans = [0] * n
    for u in reversed(order):
        ans[u] = cnt[u][lab[u]]                    # SAI: đọc trước khi cộng nhãn của chính u
        cnt[u][lab[u]] += 1
        if parent[u] != -1:
            for i in range(26):
                cnt[parent[u]][i] += cnt[u][i]
    return ans

args, want = examples[0]
print("sai:", wrong_exclude_self(*args), "| đúng:", want)

sai: [1, 0, 0, 0, 0, 0, 0] | đúng: [2, 1, 1, 1, 1, 1, 1]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Với mỗi nút, tính gì đó trên cây con của nó"** → DP trên cây theo thứ tự hậu tố (con trước cha).
- Cây cho dưới dạng **danh sách cạnh** → dựng danh sách kề hai chiều + định gốc bằng BFS.
- `n ≤ 10⁵` trong Python → **tránh đệ quy**: BFS order đảo ngược, hoặc stack `(u, vào/ra)`.
- Đếm theo nhãn trong cây con → **mẹo trước/sau** với bộ đếm toàn cục (cũng dùng cho Euler tour: cây con = đoạn liên tiếp `[tin[u], tout[u]]`).
- Bảng chữ cái nhỏ (26) → mảng đếm cố định thay vì dict.

## 15. Biến thể của bài

1. **Số nhãn khác nhau trong cây con:** OR các bitmask 26 bit (bài tập 5).
2. **LC 2003 – Smallest Missing Genetic Value in Each Subtree.**
3. **LC 1123 / 865 – Lowest Common Ancestor of Deepest Leaves** — DP trên cây.
4. **LC 2246 – Longest Path With Different Adjacent Characters** — DP trên cây với nhãn.
5. **Truy vấn "số nút nhãn c trong cây con u"** cho nhiều `(u, c)` → Euler tour + prefix count theo từng chữ cái.
6. **Kích thước cây con** (subtree size) — trường hợp đặc biệt khi mọi nhãn giống nhau.

In [25]:
# Minh hoạ biến thể 5: Euler tour -> cây con u là đoạn [tin[u], tout[u]] trong thứ tự DFS
def euler_tour(n, edges):
    g = build_adj(n, edges)
    tin, tout, order = [0] * n, [0] * n, []
    visited = [False] * n
    stack = [(0, False)]
    while stack:
        u, leaving = stack.pop()
        if leaving:
            tout[u] = len(order) - 1
            continue
        visited[u] = True
        tin[u] = len(order)
        order.append(u)
        stack.append((u, True))
        for v in g[u]:
            if not visited[v]:
                stack.append((v, False))
    return tin, tout, order

tin, tout, order = euler_tour(*examples[0][0][:2])
labels = examples[0][0][2]
print("order:", order)
for u in range(7):
    seg = order[tin[u]:tout[u] + 1]
    print(f"cây con {u}: {seg} -> cùng nhãn: {sum(labels[v] == labels[u] for v in seg)}")

order: [0, 2, 6, 3, 1, 5, 4]
cây con 0: [0, 2, 6, 3, 1, 5, 4] -> cùng nhãn: 2
cây con 1: [1, 5, 4] -> cùng nhãn: 1
cây con 2: [2, 6, 3] -> cùng nhãn: 1
cây con 3: [3] -> cùng nhãn: 1
cây con 4: [4] -> cùng nhãn: 1
cây con 5: [5] -> cùng nhãn: 1
cây con 6: [6] -> cùng nhãn: 1


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Duyệt từng cây con | `O(n²)` | `O(n)` | Đối chiếu |
| 2. DFS đệ quy | Trả mảng 26 lên cha | `O(26n)` | `O(26h)` | Ngôn ngữ có stack lớn / cây nông |
| 3. Hậu tố lặp | BFS order đảo ngược, gộp mảng 26 | `O(26n)` | `O(26n)` | An toàn, dễ hiểu |
| 4. Trước/sau | Bộ đếm toàn cục, hiệu lúc vào–ra | `O(n)` | `O(n)` | Tối ưu, gọn bộ nhớ |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().countSubTrees(3, [[1, 0], [2, 0]], "aba")` trả về gì?

### Bài 2 (tính tay)
Với Ví dụ 3 (`"aabab"`), lập bảng mảng đếm (a, b) của từng cây con từ lá lên gốc.

### Bài 3 (điền TODO)
Viết `subtree_sizes(n, edges)`: kích thước cây con của mọi nút (gốc 0), **không đệ quy**.

### Bài 4 (tự viết lại)
Viết lại Cách 4 nhưng định gốc bằng BFS trước (dùng `parent`) thay cho mảng `visited`: `count_with_parent(n, edges, labels)`.

### Bài 5 (mini challenge)
Viết `distinct_labels(n, edges, labels)`: với mỗi nút, số **nhãn khác nhau** trong cây con của nó.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def subtree_sizes(n, edges):
    # TODO: bfs_order -> duyệt ngược, size[u] = 1 + tổng size các con
    pass

# Bài 4 — TODO
def count_with_parent(n, edges, labels):
    # TODO: parent từ BFS; DFS lặp (u, vào/ra), chỉ đẩy v != parent[u]
    pass

# Bài 5 — TODO
def distinct_labels(n, edges, labels):
    # TODO: mask[u] = 1 << nhãn(u); OR mask con vào cha theo thứ tự hậu tố; đếm bit
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Gốc 0 có hai con 1, 2. (2) Cây con của 0 có nhãn `a, b, a`. (3) `[2, 1, 1]`.

**Bài 2.** (1) Lá: 3(a), 2(b), 4(b). (2) Nút 1(a) = {1, 3} → a:2. (3) Gốc: a:3, b:2 → `ans[0] = 3`.

**Bài 3.** (1) Thứ tự BFS ngược cho con trước cha. (2) `size = [1]*n`. (3) `size[parent[u]] += size[u]`.

**Bài 4.** (1) Tính `parent` bằng `bfs_order`. (2) Khi vào `u`, đẩy `(v, vào)` cho mọi `v ∈ g[u]` với `v != parent[u]`. (3) Phần còn lại giống Cách 4.

**Bài 5.** (1) 26 nhãn → bitmask 26 bit. (2) `mask[p] |= mask[u]`. (3) Số nhãn khác nhau = `bin(mask).count("1")` (hoặc `int.bit_count()`).

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1
r1 = Solution().countSubTrees(3, [[1, 0], [2, 0]], "aba")
print("Bài 1:", r1)
assert r1 == [2, 1, 1]

# Bài 2
n, edges, labels = examples[2][0]
g = build_adj(n, edges)
parent, order = bfs_order(n, g)
ab = {u: [labels[u].count("a"), labels[u].count("b")] for u in range(n)}
for u in reversed(order):
    if parent[u] != -1:
        ab[parent[u]][0] += ab[u][0]
        ab[parent[u]][1] += ab[u][1]
print("Bài 2:", ab)
assert ab[0] == [3, 2] and ab[1] == [2, 0]

Bài 1: [2, 1, 1]
Bài 2: {0: [3, 2], 1: [2, 0], 2: [0, 1], 3: [1, 0], 4: [0, 1]}


In [28]:
# Bài 3
def subtree_sizes_sol(n, edges):
    g = build_adj(n, edges)
    parent, order = bfs_order(n, g)
    size = [1] * n
    for u in reversed(order):
        if parent[u] != -1:
            size[parent[u]] += size[u]
    return size

# Bài 4
def count_with_parent_sol(n, edges, labels):
    g = build_adj(n, edges)
    parent, _ = bfs_order(n, g)
    lab = [ord(c) - 97 for c in labels]
    cnt, before, ans = [0] * 26, [0] * n, [0] * n
    stack = [(0, False)]
    while stack:
        u, leaving = stack.pop()
        if leaving:
            ans[u] = cnt[lab[u]] - before[u]
            continue
        before[u] = cnt[lab[u]]
        cnt[lab[u]] += 1
        stack.append((u, True))
        for v in g[u]:
            if v != parent[u]:
                stack.append((v, False))
    return ans

rng = random.Random(4)
for _ in range(300):
    args = random_tree(rng, rng.randint(1, 25), "abc")
    n, edges, labels = args
    assert subtree_sizes_sol(n, edges) == brute(n, edges, "a" * n)
    assert count_with_parent_sol(*args) == brute(*args)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [29]:
# Bài 5
def distinct_labels_sol(n, edges, labels):
    g = build_adj(n, edges)
    parent, order = bfs_order(n, g)
    mask = [1 << (ord(c) - 97) for c in labels]
    for u in reversed(order):
        if parent[u] != -1:
            mask[parent[u]] |= mask[u]
    return [bin(m).count("1") for m in mask]

def brute_distinct(n, edges, labels):
    g = build_adj(n, edges)
    parent, _ = bfs_order(n, g)
    res = []
    for i in range(n):
        seen, stack = set(), [i]
        while stack:
            u = stack.pop()
            seen.add(labels[u])
            stack.extend(v for v in g[u] if v != parent[u])
        res.append(len(seen))
    return res

print("Bài 5 trên Ví dụ 1:", distinct_labels_sol(*examples[0][0]))
for _ in range(300):
    args = random_tree(rng, rng.randint(1, 25), "abcdefg")
    assert distinct_labels_sol(*args) == brute_distinct(*args)
print("Đáp án Bài 1–5: mọi assert đều qua")

Bài 5 trên Ví dụ 1: [5, 3, 3, 1, 1, 1, 1]
Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template DP trên cây không đệ quy:**

```python
g = [[] for _ in range(n)]
for a, b in edges: g[a].append(b); g[b].append(a)
parent, order, seen = [-1]*n, [0], [False]*n; seen[0] = True
for u in order:
    for v in g[u]:
        if not seen[v]:
            seen[v] = True; parent[v] = u; order.append(v)
for u in reversed(order):          # con trước cha
    ... tính dp[u] từ dp các con ...
    if parent[u] != -1: gộp dp[u] vào dp[parent[u]]
```

**Template trước/sau (vào/ra):** `before[u] = cnt[c]` khi vào, `cnt[c] += 1`; khi ra `ans[u] = cnt[c] − before[u]`.

**Checklist trước khi submit:**
- [ ] Danh sách kề **hai chiều**, không giả định chiều cạnh.
- [ ] Bỏ qua cha / dùng `visited` khi duyệt.
- [ ] Không đệ quy sâu (cây đường thẳng `10⁵`).
- [ ] Không tạo mảng bằng `[[0]*26]*n`.
- [ ] `ans[i]` tính cả chính nút `i`.
- [ ] Đúng chữ ký: `countSubTrees(self, n: int, edges: list[list[int]], labels: str) -> list[int]`.